# VisionAI: train the road-hazard detector

Fine-tunes YOLOv8n on Indian road hazards (potholes, stray animals, construction barriers, open drains...).

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. In Google Drive create `MyDrive/VisionAI/` containing:
   - `training/` : the `training` folder from the project zip
   - `raw_datasets/` : the dataset zips you downloaded (Roboflow: *Download dataset > YOLOv8*, plus `RDD2022_India.zip`, DriveIndia, etc.)
   - `yolov8n.pt` *(optional)*: your weights; otherwise the official ones are downloaded
3. Run the cells top to bottom.

In [ ]:
# ---- SETTINGS (edit these) ----
PROJECT = "/content/drive/MyDrive/VisionAI"
MODEL   = "yolov8n.pt"      # or "yolo26n.pt" to try the newer edge-oriented model
EPOCHS  = 100               # do a quick 30-50 epoch trial first
IMGSZ   = 640            # training / laptop size
PI_IMGSZ = 320          # Raspberry Pi export size: MUST equal imgsz= in rpi_test.py
BATCH   = 16
RUN     = "visionai_v1"

In [ ]:
!nvidia-smi | head -15
!pip -q install ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
if os.path.exists(f"{PROJECT}/{MODEL}"):
    shutil.copy(f"{PROJECT}/{MODEL}", f"/content/{MODEL}")   # use your own copy of the weights

## 1. Unpack datasets
Every `.zip` in `raw_datasets/` is extracted to `/content/raw/<zip name>/`. The zip name is what you use as the key under `overrides:` in `class_map.yaml`.

In [ ]:
import zipfile, pathlib
raw = pathlib.Path("/content/raw"); raw.mkdir(exist_ok=True)
for z in sorted(pathlib.Path(f"{PROJECT}/raw_datasets").glob("*.zip")):
    dest = raw / z.stem
    if dest.exists():
        print("already unpacked:", z.name); continue
    print("unpacking", z.name)
    zipfile.ZipFile(z).extractall(dest)
print(sorted(p.name for p in raw.iterdir()))

## 2. (Optional) RDD2022 India: keep only potholes
Skip this cell if you didn't download RDD2022. It converts the Pascal-VOC XML to YOLO format.

In [ ]:
import pathlib
rdd = pathlib.Path("/content/raw/RDD2022_India")
if rdd.exists():
    !python {PROJECT}/training/voc_to_yolo.py --src {rdd} --out /content/raw/rdd_india_potholes
else:
    print("RDD2022_India.zip not found, skipping.")

## 3. Merge everything into one dataset
Read the report: it lists every source class that was **dropped**. If something important is listed, add it to `class_map.yaml` (aliases or overrides), re-upload it to Drive, and re-run.

In [ ]:
!python {PROJECT}/training/merge_datasets.py --raw /content/raw --out /content/datasets/visionai --map {PROJECT}/training/class_map.yaml --clean

## 4. Look at the labels (don't skip)
A wrong class mapping silently ruins a model. Check that boxes sit on the right objects and carry the right names.

In [ ]:
DATA = "/content/datasets/visionai/data.yaml"
!python {PROJECT}/training/preview_labels.py --data {DATA} --n 12 --out /content/preview
from IPython.display import display
from PIL import Image
import glob
for p in sorted(glob.glob("/content/preview/*"))[:8]:
    im = Image.open(p); im.thumbnail((640, 640)); display(im)

In [ ]:
# Spot-check the rare, safety-critical classes
for cls in ["pothole", "animal", "open_manhole_drain", "barricade"]:
    !python {PROJECT}/training/preview_labels.py --data {DATA} --n 4 --only {cls} --out /content/preview_{cls}

## 5. Train
Saved straight to Drive (`VisionAI/runs/`) so a Colab disconnect doesn't lose progress.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=20,
            project=f"{PROJECT}/runs", name=RUN, exist_ok=True)

**If Colab disconnected mid-training:** re-run the install, mount, unpack, RDD, merge cells (the local dataset is wiped on reset), then run:
```python
from ultralytics import YOLO
YOLO(f"{PROJECT}/runs/{RUN}/weights/last.pt").train(resume=True)
```

## 6. Evaluate per class
Overall mAP can hide a useless pothole detector. Look at each class.

In [ ]:
from ultralytics import YOLO
best = f"{PROJECT}/runs/{RUN}/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=DATA, imgsz=IMGSZ)
print(f"\nOverall  mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}\n")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{m.names[int(c)]:<22} mAP50 {metrics.box.ap50[i]:.3f}   mAP50-95 {metrics.box.maps[int(c)]:.3f}")

In [ ]:
# Quick visual test on a few validation images
import glob
from IPython.display import display
from PIL import Image
imgs = sorted(glob.glob("/content/datasets/visionai/images/val/*"))[:8]
m.predict(imgs, conf=0.4, imgsz=IMGSZ, save=True, project="/content/pred", name="val", exist_ok=True, verbose=False)
for p in sorted(glob.glob("/content/pred/val/*"))[:8]:
    im = Image.open(p); im.thumbnail((640, 640)); display(im)

## 7. Export for the Raspberry Pi
`best.pt` is already in Drive. The NCNN export is the faster format for the Pi (it creates a `best_ncnn_model/` folder next to it).

In [ ]:
m.export(format="ncnn", imgsz=PI_IMGSZ)
print("Download from Drive:", f"{PROJECT}/runs/{RUN}/weights/")
print("  best.pt            -> laptop (src/test.py)")
print(f"  best_ncnn_model/   -> Raspberry Pi (src/rpi_test.py, set imgsz={PI_IMGSZ} there)")